# 1.0 Exploratory Data Analysis: Pediatric Chest Radiographs

**Project:** Healthcare & Medical – Diagnostic Lab Image Classification  
**Organization:** Axlero  
**Milestone:** Phase 1 (Day 5) – Initial Exploratory Data Analysis & Visualizations  
**Branch:** `healthcare-cnn-development`  

---

## 1. Objectives & Executive Summary
1. Load and catalog all raw chest X-ray images across `train`, `val`, and `test` partitions.
2. Evaluate class balance, frequency disparities, and clinical implications.
3. Interactively compare representative radiographs for **NORMAL**, **BACTERIAL**, and **VIRAL** pneumonia.
4. Analyze geometric heterogeneity (width, height, aspect ratio distributions).
5. Establish engineering requirements for CNN transfer learning (ResNet50 / DenseNet121).


In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

project_root = Path('..').resolve() if Path('..').resolve().name != 'notebooks' else Path('.').resolve()
data_dir = project_root / 'data' / 'raw' / 'chest_xray'
print(f"Project Root: {project_root}")
print(f"Data Directory: {data_dir}")
assert data_dir.exists(), f"Data directory not found at {data_dir}"


## 2. Dataset Inventory & Class Frequencies
Scanning the directory hierarchy to compile inventory metrics across all splits.

In [ ]:
splits = ['train', 'val', 'test']
records = []

for split in splits:
    split_path = data_dir / split
    for cls_path in sorted(split_path.iterdir()):
        if cls_path.is_dir():
            img_files = list(cls_path.glob('*.jpeg'))
            for img_f in img_files:
                records.append({
                    'split': split,
                    'class': cls_path.name,
                    'filename': img_f.name,
                    'path': str(img_f)
                })

df = pd.DataFrame(records)
print(f"Total Radiographs Indexed: {len(df):,}")
summary_table = pd.crosstab(df['split'], df['class'], margins=True)
summary_table['% Pneumonia'] = (summary_table['PNEUMONIA'] / summary_table['All'] * 100).round(2)
display(summary_table)


## 3. Class Distribution Visualizations
Visualizing class frequencies and proportions across splits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
split_counts = df.groupby(['split', 'class']).size().unstack(fill_value=0)
split_counts.plot(kind='bar', ax=axes[0], color=['#2b5c8f', '#d95f02'], edgecolor='black', width=0.7)
axes[0].set_title('Radiograph Frequency by Split', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Dataset Split')
axes[0].set_ylabel('Number of Images')
axes[0].legend(title='Diagnosis')

total_counts = df['class'].value_counts()
axes[1].pie(
    total_counts.values,
    labels=[f"{cls}\n({cnt:,})" for cls, cnt in total_counts.items()],
    autopct='%1.1f%%',
    colors=['#d95f02', '#2b5c8f'],
    startangle=140,
    explode=(0.04, 0),
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2}
)
axes[1].set_title(f'Overall Class Balance (N = {len(df):,})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Clinical Visual Comparison: NORMAL vs BACTERIAL vs VIRAL
- **NORMAL:** Clear bilateral lung aeration, distinct diaphragmatic contours.
- **BACTERIAL PNEUMONIA:** Dense focal consolidation (often lobar) with air bronchograms.
- **VIRAL PNEUMONIA:** Diffuse bilateral interstitial or peribronchial opacities.

In [ ]:
train_normal = df[(df['split'] == 'train') & (df['class'] == 'NORMAL')]['path'].tolist()[:3]
train_bacterial = [p for p in df[(df['split'] == 'train') & (df['class'] == 'PNEUMONIA')]['path'] if 'bacteria' in p.lower()][:3]
train_viral = [p for p in df[(df['split'] == 'train') & (df['class'] == 'PNEUMONIA')]['path'] if 'virus' in p.lower()][:3]

categories = [
    ('NORMAL (Clear Lungs)', train_normal, '#2b5c8f'),
    ('PNEUMONIA (Bacterial - Focal)', train_bacterial, '#c0392b'),
    ('PNEUMONIA (Viral - Interstitial)', train_viral, '#e67e22')
]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row_idx, (title, sample_paths, col) in enumerate(categories):
    for col_idx, p in enumerate(sample_paths):
        ax = axes[row_idx, col_idx]
        with Image.open(p) as img:
            ax.imshow(img.convert('RGB'))
            ax.set_title(f"{Path(p).name[:15]}... ({img.size[0]}x{img.size[1]})", fontsize=9)
        ax.axis('off')
    axes[row_idx, 0].text(-0.1, 0.5, title, transform=axes[row_idx, 0].transAxes,
                          fontsize=11, fontweight='bold', color=col, va='center', ha='right', rotation=90)

plt.suptitle('Clinical Radiograph Visual Comparison', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 5. Geometric Distribution Analysis
Assessing pixel width and height distributions to configure PyTorch resize transforms.

In [ ]:
sample_df = df.sample(min(500, len(df)), random_state=42).copy()
dims = []
for p in sample_df['path']:
    with Image.open(p) as img:
        dims.append(img.size)

sample_df['width'] = [d[0] for d in dims]
sample_df['height'] = [d[1] for d in dims]
sample_df['aspect_ratio'] = sample_df['width'] / sample_df['height']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(sample_df['width'], kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title(f"Width (Mean: {sample_df['width'].mean():.0f}px)", fontweight='bold')

sns.histplot(sample_df['height'], kde=True, ax=axes[1], color='#d95f02')
axes[1].set_title(f"Height (Mean: {sample_df['height'].mean():.0f}px)", fontweight='bold')

sns.histplot(sample_df['aspect_ratio'], kde=True, ax=axes[2], color='#27ae60')
axes[2].set_title(f"Aspect Ratio (Mean: {sample_df['aspect_ratio'].mean():.2f})", fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Preprocessing Directives for CNN Modeling
1. Standardize all input images to **224x224 RGB tensors**.
2. Apply **training-only dynamic data augmentation** (rotation ±10°, horizontal flip, contrast/brightness jitter).
3. Apply **deterministic evaluation transforms** for validation and test splits.
4. Restructure the validation set from 16 images to a statistically reliable 10%-15% split.
5. Calculate positive class weighting in binary cross-entropy loss to drive recall > 90%.
